# Introdução a GNNs com PyTorch Geometric

> Parte da série [ML Notebooks](../README.md) — por **Nandobez**.


Neste notebook curto, o objetivo é dar um guia introdutório para começar com Graph Neural Networks usando a biblioteca [PyTorch Geometric](https://pytorch-geometric.readthedocs.io/en/latest/index.html). Como é baseada em PyTorch, vamos usar PyTorch ao longo do tutorial.

O código foi adaptado dos [exemplos oficiais](https://pytorch-geometric.readthedocs.io/en/latest/notes/introduction.html). Acrescentei explicações mais amigáveis para iniciantes e mantive minimal.


In [ ]:
# Find the CUDA version PyTorch was installed with
!python -c "import torch; print(torch.version.cuda)"

In [ ]:
# PyTorch version
!python -c "import torch; print(torch.__version__)"

Instale os pacotes a seguir, mas atenção à versão correta. Veja as [instruções](https://pytorch-geometric.readthedocs.io/en/latest/notes/installation.html) se tiver dúvidas.


In [ ]:
!pip install torch-scatter -f https://data.pyg.org/whl/torch-1.11.0.html

In [ ]:
!pip install torch-sparse -f https://data.pyg.org/whl/torch-1.11.0.html

In [ ]:
!pip install torch-geometric

## Primeiros Passos

Importar o PyTorch.


In [ ]:
import torch

# print torch version
print(torch.__version__)

A vantagem do PyTorch Geometric é trazer funcionalidades para importar e carregar dados em formato de grafo.


In [ ]:
from torch_geometric.data import Data

Vamos criar um grafo não ponderado e não direcionado com três nós e quatro arestas no total.


In [ ]:
# define edge list
edge_index = torch.tensor([[0, 1, 1, 2], [1, 0, 2, 1]], dtype=torch.long)

# define node features
x = torch.tensor([[-1], [0], [1]])

# create graph data object
data = Data(x=x, edge_index=edge_index)
print(data)

O objeto `Data` tem várias funções utilitárias para inspecionar propriedades do grafo.


In [ ]:
# check number of edges of the graph
print(data.num_edges)

In [ ]:
# check number of nodes of the graph
print(data.num_nodes)

In [ ]:
# check number of features of the graph
print(data.num_features)

In [ ]:
# check if graph is directed
print(data.is_directed())

## Carregando Dados

Veja mais funções úteis sobre grafos [aqui](https://pytorch-geometric.readthedocs.io/en/latest/modules/data.html#torch_geometric.data.Data).

Uma coisa bacana do PyTorch Geometric é que ele inclui conjuntos de benchmark prontos. Um popular é o Cora, usado para classificação supervisionada de nós em grafos.

> "O conjunto Cora consiste em 2708 publicações científicas classificadas em uma de sete classes. A rede de citação tem 5429 links. Cada publicação é descrita por um vetor de palavras 0/1, indicando presença/ausência de cada palavra de um dicionário de 1433 termos." — [Papers with Code](https://paperswithcode.com/dataset/cora)

Vamos carregar o Cora:


In [ ]:
from torch_geometric.datasets import Planetoid

dataset = Planetoid(root='tmp/Cora', name='Cora')

Vamos inspecionar propriedades do Cora.


In [ ]:
# number of graphs
print("Number of graphs: ", len(dataset))

# number of features
print("Number of features: ", dataset.num_features)

# number of classes
print("Number of classes: ", dataset.num_classes)

Esse conjunto contém apenas um grafo. Dados de grafos podem ser complexos e incluir múltiplos grafos. Vamos ver mais features do Cora:


In [ ]:
# select the first graph
data = dataset[0]

# number of nodes
print("Number of nodes: ", data.num_nodes)

# number of edges
print("Number of edges: ", data.num_edges)

# check if directed
print("Is directed: ", data.is_directed())

Você pode amostrar nós do grafo assim:


In [ ]:
# sample nodes from the graph
print("Shape of sample nodes: ", data.x[:5].shape)

Extraímos 5 nós do grafo e checamos a shape: cada nó tem `1433` features.

Outra vantagem: o Cora vem pré-processado e pronto para uso, com splits de treino, validação e teste já definidos.


In [ ]:
# check training nodes
print("# of nodes to train on: ", data.train_mask.sum().item())

# check test nodes
print("# of nodes to test on: ", data.test_mask.sum().item())

# check validation nodes
print("# of nodes to validate on: ", data.val_mask.sum().item())

Essa informação é importante para o modelo: indica em quais nós treinar e em quais avaliar.

Ao treinar redes neurais usamos batches. PyTorch Geometric tem `DataLoader` específico para grafos.


In [ ]:
from torch_geometric.datasets import Planetoid
from torch_geometric.loader import DataLoader
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
print(device)

In [ ]:
dataset = Planetoid(root='tmp/Cora', name='Cora')
data = dataset[0].to(device)

Algumas estatísticas rápidas sobre os dados:


In [ ]:
print("X shape: ", data.x.shape)
print("Edge shape: ", data.edge_index.shape)
print("Y shape: ", data.y.shape)

## Modelo e Treinamento

Finalmente, definimos uma GCN padrão para treinar no Cora. Objetivo: melhorar a predição da classe de cada nó.

Para manter simples, usamos a mesma definição do [tutorial original](https://pytorch-geometric.readthedocs.io/en/latest/notes/introduction.html). Note o uso do `GCNConv` nativo — mas você pode implementar do zero.

O modelo abaixo tem duas camadas `GCNConv`. A primeira é seguida de `ReLU` + `Dropout`. O resultado vai para a segunda, na qual aplicamos `Softmax` sobre o número de classes.


In [ ]:
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class GCN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        """ GCNConv layers """
        self.conv1 = GCNConv(data.num_features, 16)
        self.conv2 = GCNConv(16, dataset.num_classes)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, training=self.training)
        x = self.conv2(x, edge_index)

        return F.log_softmax(x, dim=1)

Inicializar modelo e otimizador.


In [ ]:
model = GCN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

Função de acurácia para avaliação:


In [ ]:
# useful function for computing accuracy
def compute_accuracy(pred_y, y):
    return (pred_y == y).sum()

Treino do modelo nos nós de treino por 200 épocas:


In [ ]:
# train the model
model.train()
losses = []
accuracies = []
for epoch in range(200):
    optimizer.zero_grad()
    out = model(data)

    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
    correct = compute_accuracy(out.argmax(dim=1)[data.train_mask], data.y[data.train_mask])
    acc = int(correct) / int(data.train_mask.sum())
    losses.append(loss.item())
    accuracies.append(acc)

    loss.backward()
    optimizer.step()
    if (epoch+1) % 10 == 0:
        print('Epoch: {}, Loss: {:.4f}, Training Acc: {:.4f}'.format(epoch+1, loss.item(), acc))


In [ ]:
# plot the loss and accuracy
import matplotlib.pyplot as plt
plt.plot(losses)
plt.plot(accuracies)
plt.legend(['Loss', 'Accuracy'])
plt.show()

Parece que o modelo atinge acurácia alta e perda baixa no treino. Vamos testar nos nós de teste para verificar a generalização:


In [ ]:
# evaluate the model on test set
model.eval()
pred = model(data).argmax(dim=1)
correct = compute_accuracy(pred[data.test_mask], data.y[data.test_mask])
acc = int(correct) / int(data.test_mask.sum())
print(f'Accuracy: {acc:.4f}')

Boa acurácia também no teste — o modelo está indo bem. Existem várias formas de melhorar, mas deixamos para outro momento.

Não testei o código em GPU — fica como exercício.


## References

- Series repo: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Author: [Nandobez](https://github.com/Nandobez)
